# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [ ]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, trim, upper
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: b6729cc9-c5f6-47ec-b463-5ac69788f0ac
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session b6729cc9-c5f6-47ec-b463-5ac69788f0ac to get into ready status...
Session b6729cc9-c5f6-47ec-b463-5ac69788f0ac 

#### Extraindo arquivo do s3


In [ ]:
path_bronze = "s3://datalake-404604958697/bronze/dicionario_pnad/"

df_bronze = spark.read.parquet(path_bronze)

df_bronze.show(5)
df_bronze.printSchema()

+------------+--------------------+-------------+-------------------+--------------+
|cod_variavel|  descricao_variavel|cod_categoria|descricao_categoria|  id_categoria|
+------------+--------------------+-------------+-------------------+--------------+
|          UF|Unidade da Federação|           11|           Rondônia|UF_11_Rondônia|
|          UF|Unidade da Federação|           12|               Acre|    UF_12_Acre|
|          UF|Unidade da Federação|           13|           Amazonas|UF_13_Amazonas|
|          UF|Unidade da Federação|           14|            Roraima| UF_14_Roraima|
|          UF|Unidade da Federação|           15|               Pará|    UF_15_Pará|
+------------+--------------------+-------------+-------------------+--------------+
only showing top 5 rows

root
 |-- cod_variavel: string (nullable = true)
 |-- descricao_variavel: string (nullable = true)
 |-- cod_categoria: string (nullable = true)
 |-- descricao_categoria: string (nullable = true)
 |-- id_categor

#### Transformações

In [ ]:
df_silver = df_bronze \
    .withColumn("cod_variavel", upper(trim(col("cod_variavel")))) \
    .withColumn("descricao_variavel", trim(col("descricao_variavel"))) \
    .withColumn("cod_categoria", trim(col("cod_categoria")).cast("string")) \
    .withColumn("descricao_categoria", trim(col("descricao_categoria")))

# remover possíveis registros inválidos
df_silver = df_silver.filter(col("cod_categoria").isNotNull())

# remover duplicados
df_silver = df_silver.dropDuplicates()

df_silver.show(5)
df_silver.printSchema()

+------------+--------------------+--------------------+--------------------+--------------------+
|cod_variavel|  descricao_variavel|       cod_categoria| descricao_categoria|        id_categoria|
+------------+--------------------+--------------------+--------------------+--------------------+
|          UF|Unidade da Federação|                  23|               Ceará|         UF_23_Ceará|
|     CAPITAL|             Capital|                  22|Município de Tere...|CAPITAL_22_Municí...|
|     RM_RIDE|Região Metropolit...|                  16|Região Metropolit...|RM_RIDE_16_Região...|
|      POSEST|Domínios de projeção|6 dígitos e 8 cas...|As 2 primeiras po...|posest_6 dígitos ...|
|        A005|        Escolaridade|                   2|Fundamental incom...|A005_2_Fundamenta...|
+------------+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows

root
 |-- cod_variavel: string (nullable = true)
 |-- descricao_variavel: string (nu

#### Example: Salvando dados no S3


In [ ]:
path_silver = "s3://datalake-404604958697/silver/dicionario_covid/"

df_silver.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(path_silver)